# PTM-Prediction — Fases 1-3b

Corre las fases GPU-pesadas del pipeline (**1 → 1.5 → 2 → 3 → 3b**).

**Qué corre:** saneamiento, extracción de estructura, DeepMVP, DeepPTMPred, y los
cruces informativos de fase 3b (vía secretora, Kinase Library, MeToken, EMNGly,
competencia entre PTMs).

**Qué no corre:**
- **Fase 3c** (modelado estructural real con PyRosetta): es CPU-only, no se beneficia de
  la GPU de Colab — se desactiva con `FASE_A_ENABLED=false` y se recomienda seguir
  corriéndola local, acotada (~9 sitios máx, minutos cada uno).
- **StackGlyEmbed**: reutiliza el venv de un proyecto hermano
  (`B-Cell-Epitope-Prediction/StackGlyEmbed`) + pesos de ProtT5 (~3GB) de un tercer
  proyecto (`scipion-chem-tmbed`). Traerlos a Colab implica clonar y pesar dos repos más
  no relacionados con este.
  (`STACKGLYEMBED_ENABLED=false`) — el consenso de N-glicosilación sigue funcionando con
  DeepMVP + EMNGly (2 motores, alcanza el mínimo de consenso).


In [ ]:
!nvidia-smi


## 1. Google Drive (cache persistente de pesos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
for sub in ["weights/DeepMVP/models",
            "weights/DeepPTMPred/esm",
            "weights/MeToken",
            "weights/EMNgly/esm",
            "weights/EMNgly/checkpoints",
            "outputs_backup"]:
    os.makedirs(f"{CACHE_ROOT}/{sub}", exist_ok=True)
print("Cache persistente en:", CACHE_ROOT)


## 2. `condacolab` (Miniforge + `mamba`)

Esta celda **reinicia el runtime de Python automáticamente** (comportamiento normal de
`condacolab`, no es un error) — el mount de Drive sobrevive el reinicio porque es un
mount FUSE a nivel de sistema operativo, no estado de Python. Después del reinicio,
seguí ejecutando la celda siguiente con normalidad.


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


### Reanudar acá después del reinicio automático

In [ ]:
import condacolab
condacolab.check()
!mamba --version

import os
CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
REPO = "/content/PTM-Prediction"
assert os.path.isdir(CACHE_ROOT), "Drive no esta montado -- volve a correr la celda de mount."


## 3. Clonar el pipeline principal + los 4 repos de motores + deps livianas

In [ ]:
%cd /content
!git clone -q https://github.com/Lvera-code/PTM-Prediction.git

%cd {REPO}
!pip install -q -r requirements.txt

# Los 4 repos de motores se clonan todos de una, ANTES de crear ninguna subcarpeta de
# checkpoints -- `git clone` se niega a clonar sobre un directorio ya existente y no
# vacio, asi que el orden acá importa.
!git clone -q https://github.com/bzhanglab/DeepMVP DeepMVP
!git clone -q https://github.com/kuikui-wang/DeepPTMPred DeepPTMPred
!git clone -q https://github.com/A4Bio/MeToken MeToken
!git clone -q https://github.com/StellaHxy/EMNgly EMNgly
print("5 repos clonados.")


## 4. Arrancar en segundo plano las descargas pesadas

`aria2c` con 16 conexiones por archivo, corriendo en background (`&` + `disown`) mientras
las celdas siguientes crean los entornos conda -- se solapan en vez de esperar una cosa
después de la otra. Si Drive ya tiene el archivo de una sesión anterior, se copia directo
(segundos) en vez de re-descargar. La celda de la Sección 12 espera de verdad (polling de
los logs) antes de correr el pipeline.


In [ ]:
!apt-get -qq install -y aria2 > /dev/null
import os
os.makedirs("/content/dl_logs", exist_ok=True)
DL_LOGS = [
    "/content/dl_logs/esm2_main.log",
    "/content/dl_logs/esm2_reg.log",
    "/content/dl_logs/esm1b_main.log",
    "/content/dl_logs/esm1b_reg.log",
    "/content/dl_logs/nglyde_svm.log",
    "/content/dl_logs/metoken_zip.log",
]
for f in DL_LOGS:
    open(f, "w").close()


In [ ]:
%%bash -s "$CACHE_ROOT" "$REPO"
CACHE_ROOT="$1"
REPO="$2"

fetch_or_copy () {
  # $1 = etiqueta, $2 = ruta en cache Drive, $3 = URL, $4 = destino final, $5 = log
  local tag="$1" drive_path="$2" url="$3" dest="$4" log="$5"
  mkdir -p "$(dirname "$dest")"
  if [ -s "$drive_path" ]; then
    echo "[$tag] ya en Drive, copiando..." > "$log"
    cp "$drive_path" "$dest"
  else
    echo "[$tag] descargando con aria2c..." > "$log"
    aria2c -x 16 -s 16 -k 1M -q -o "$(basename "$dest")" -d "$(dirname "$dest")" "$url" >> "$log" 2>&1
    mkdir -p "$(dirname "$drive_path")"
    cp "$dest" "$drive_path"
  fi
  echo "[$tag] listo." >> "$log"
}

# ESM-2 650M (DeepPTMPred, ~2.6GB) + companero de regresion de contactos
fetch_or_copy "esm2-main" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D.pt" \
  "/content/dl_logs/esm2_main.log" &

fetch_or_copy "esm2-reg" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt" \
  "/content/dl_logs/esm2_reg.log" &

# ESM-1b 650M (EMNGly, ~7.4GB) + companero de regresion de contactos
fetch_or_copy "esm1b-main" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S.pt" \
  "/content/dl_logs/esm1b_main.log" &

fetch_or_copy "esm1b-reg" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "/content/dl_logs/esm1b_reg.log" &

# SVM N-GlyDE.pickle (EMNGly, ~36MB, Google Drive publico del autor)
fetch_or_copy "nglyde-svm" \
  "$CACHE_ROOT/weights/EMNgly/checkpoints/N-GlyDE.pickle" \
  "https://drive.usercontent.google.com/download?id=1hbnEtHHXTGnQAFm-cCHMj3pWQiAYAUsw&export=download&confirm=t" \
  "$REPO/EMNgly/checkpoints/N-GlyDE.pickle" \
  "/content/dl_logs/nglyde_svm.log" &

# MeToken pretrained_model.zip (release 1.0, ~88MB) -- no usa fetch_or_copy porque hay
# que descomprimirlo, no solo copiarlo
(
  if [ -s "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" ]; then
    echo "[metoken-zip] ya en Drive, copiando..." > /content/dl_logs/metoken_zip.log
    cp "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" /tmp/pretrained_model.zip
  else
    echo "[metoken-zip] descargando..." > /content/dl_logs/metoken_zip.log
    curl -sL -o /tmp/pretrained_model.zip \
      https://github.com/A4Bio/MeToken/releases/download/1.0/pretrained_model.zip
    cp /tmp/pretrained_model.zip "$CACHE_ROOT/weights/MeToken/pretrained_model.zip"
  fi
  echo "[metoken-zip] listo." >> /content/dl_logs/metoken_zip.log
) &

disown -a
echo "6 descargas arrancaron en background, desacopladas de esta celda."
echo "Segui con las celdas siguientes -- las descargas avanzan en paralelo."
echo "(Para ver progreso en cualquier momento: !tail -n2 /content/dl_logs/*.log)"


## 5. DeepMVP (Fase 2, motor único Camino FASTA / motor 1 de 2 Camino PDB)

Python 3.7.10 + TensorFlow 2.4.2 -- entorno propio via `environment.yml` del repo real
(no reinventado). Intenta GPU (`cudatoolkit=11.0`/`cudnn=8.0` conda-forge, corre bien
sobre el driver moderno de Colab por compatibilidad hacia atrás); si TF igual no ve la
GPU en este stack tan viejo, DeepMVP sigue funcionando en CPU sin romper el resto del
pipeline -- no es el motor más pesado del consenso.

**Los pesos de DeepMVP NO son descargables por URL directa** (repositorio Shiny detrás
de https://deepmvp.ptmax.org/, requiere click manual). Si es la primera vez:
1. Descargalos una vez desde ese sitio (`models.tar.gz`, ~1.6GB) o reusá el
   `DeepMVP/models/` que ya tenés local en tu máquina.
2. Subilos a Drive UNA sola vez a `PTM-Prediction-Colab/weights/DeepMVP/models/`
   (Drive desktop, o subida manual por la interfaz web).
Las siguientes sesiones ya los encuentran en Drive.


In [ ]:
%cd {REPO}
!mamba env create -q -f DeepMVP/environment.yml -n deepmvp
DEEPMVP_PYTHON_BIN = !mamba run -n deepmvp which python
DEEPMVP_PYTHON_BIN = DEEPMVP_PYTHON_BIN[0]
print("DEEPMVP_PYTHON_BIN =", DEEPMVP_PYTHON_BIN)


In [ ]:
import os, shutil
drive_models = f"{CACHE_ROOT}/weights/DeepMVP/models"
local_models = f"{REPO}/DeepMVP/models"

if os.listdir(drive_models):
    print("Copiando pesos de DeepMVP desde Drive...")
    shutil.copytree(drive_models, local_models, dirs_exist_ok=True)
    print("Listo:", os.listdir(local_models))
else:
    print("!! No hay pesos de DeepMVP en Drive todavia.")
    print(f"!! Subi 'models/' (descargado de https://deepmvp.ptmax.org/) a:")
    print(f"!!   {drive_models}")
    print("!! y volve a correr esta celda. Mientras tanto, DeepMVP queda sin pesos")
    print("!! (el engine degrada solo con un aviso, no tumba el resto del pipeline).")


## 6. DeepPTMPred (Fase 2, motor 2 de 2 del consenso -- Camino PDB)

Python 3.10 + TensorFlow 2.15 + PyTorch 2.0 + `fair-esm`, `cudatoolkit=11.8` real (GPU).
Los pesos `.h5` por tipo de PTM ya vienen incluidos en el clon del repo -- solo falta el
checkpoint ESM-2 (descargándose en background desde la Sección 4).


In [ ]:
%cd {REPO}
!mamba env create -q -f DeepPTMPred/pred/train_PTM/environment.yml -n deepptmpred
DEEPPTMPRED_PYTHON_BIN = !mamba run -n deepptmpred which python
DEEPPTMPRED_PYTHON_BIN = DEEPPTMPRED_PYTHON_BIN[0]
print("DEEPPTMPRED_PYTHON_BIN =", DEEPPTMPRED_PYTHON_BIN)


## 7. MeToken (corroboración opcional de tipo -- Camino PDB)

Python 3.10 + PyTorch con build **CUDA** (cu121, en vez del CPU-only que usaba el
`README.md` local por no tener GPU) + `torch_scatter` como **wheel prebuilt** de PyG
matcheado a la versión de torch/cuda instalada (en vez de compilar desde fuente, que es
lo que hacía falta localmente sin GPU) -- con fallback automático a compilar si no hay
wheel exacto disponible.


In [ ]:
%cd {REPO}
!mamba create -q -n metoken python=3.10 -y
!mamba run -n metoken pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!mamba run -n metoken pip install -q numpy scipy biopython transformers omegaconf tqdm pandas huggingface-hub h5py

!mamba run -n metoken pip install -q torch_scatter -f https://data.pyg.org/whl/torch-2.4.0+cu121.html \
  || (echo "Wheel prebuilt no disponible para esta version -- compilando desde fuente (mas lento)." \
      && mamba run -n metoken pip install -q torch_scatter)

METOKEN_PYTHON_BIN = !mamba run -n metoken which python
METOKEN_PYTHON_BIN = METOKEN_PYTHON_BIN[0]
print("METOKEN_PYTHON_BIN =", METOKEN_PYTHON_BIN)


## 8. EMNGly (motor real de consenso de N-glicosilación -- Camino PDB)

venv (no conda) + `fair-esm`/torch **CUDA** + `scikit-learn==1.1.1` fijado exacto (los
pickles del SVM se entrenaron con esa versión -- una más nueva dispara
`InconsistentVersionWarning`) + reinstalar `numpy==1.23.5` después, mismo bug real ya
documentado en el `README.md` del proyecto (pip resuelve numpy 2.x por defecto,
incompatible en runtime con el wheel de sklearn 1.1.1).


In [ ]:
%cd {REPO}
!python3 -m venv .venv-emngly
!.venv-emngly/bin/pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!.venv-emngly/bin/pip install -q fair-esm "scikit-learn==1.1.1" scipy pandas tqdm wget
!.venv-emngly/bin/pip install -q "numpy==1.23.5"
EMNGLY_PYTHON_BIN = f"{REPO}/.venv-emngly/bin/python"
print("EMNGLY_PYTHON_BIN =", EMNGLY_PYTHON_BIN)


## 9. Kinase Library (corroboración informativa de fosforilación -- ambos caminos)

Paquete PyPI liviano, sin pesos grandes que descargar -- entorno separado porque fija
`numpy`/`pandas` incompatibles con el resto del pipeline.


In [ ]:
!mamba create -q -n kinase_library python=3.10 -y
!mamba run -n kinase_library pip install -q kinase-library
KINASE_LIBRARY_PYTHON_BIN = !mamba run -n kinase_library which python
KINASE_LIBRARY_PYTHON_BIN = KINASE_LIBRARY_PYTHON_BIN[0]
print("KINASE_LIBRARY_PYTHON_BIN =", KINASE_LIBRARY_PYTHON_BIN)


## 10. Variables de entorno del pipeline

In [ ]:
import os

os.environ["DEEPMVP_PYTHON_BIN"] = DEEPMVP_PYTHON_BIN
os.environ["DEEPMVP_HOME"] = f"{REPO}/DeepMVP"
os.environ["DEEPMVP_MODEL_DIR"] = f"{REPO}/DeepMVP/models"

os.environ["DEEPPTMPRED_PYTHON_BIN"] = DEEPPTMPRED_PYTHON_BIN
os.environ["DEEPPTMPRED_HOME"] = f"{REPO}/DeepPTMPred"

os.environ["METOKEN_PYTHON_BIN"] = METOKEN_PYTHON_BIN
os.environ["METOKEN_HOME"] = f"{REPO}/MeToken"
os.environ["METOKEN_ENABLED"] = "true"

os.environ["EMNGLY_PYTHON_BIN"] = EMNGLY_PYTHON_BIN
os.environ["EMNGLY_HOME"] = f"{REPO}/EMNgly"
os.environ["EMNGLY_ENABLED"] = "true"

os.environ["KINASE_LIBRARY_PYTHON_BIN"] = KINASE_LIBRARY_PYTHON_BIN
os.environ["KINASE_LIBRARY_ENABLED"] = "true"

# StackGlyEmbed y Fase 3c (PyRosetta) deliberadamente afuera de esta corrida -- ver
# Seccion 10 y la introduccion del notebook.
os.environ["STACKGLYEMBED_ENABLED"] = "false"
os.environ["FASE_A_ENABLED"] = "false"

os.environ["FASTA_INPUT_DIR"] = f"{REPO}/inputs"
os.environ["FASTA_OUTPUT_DIR"] = f"{REPO}/outputs"

print("Variables de entorno seteadas.")


## 11. Esperar que terminen las descargas en background

Espera real por polling de los logs de la Sección 4 (no `!wait` -- cada celda de shell
es un proceso nuevo sin visibilidad de los jobs de una celda anterior). Si ya venís de
una sesión previa con todo cacheado en Drive, esto termina en segundos.


In [ ]:
import time

def wait_for_downloads(logs, poll=15, timeout=3600):
    start = time.time()
    pending = set(logs)
    while pending:
        done_now = set()
        for log in pending:
            try:
                if "listo." in open(log).read():
                    done_now.add(log)
            except FileNotFoundError:
                pass
        pending -= done_now
        if not pending:
            break
        if time.time() - start > timeout:
            raise TimeoutError(f"Timeout esperando: {pending}")
        elapsed = int(time.time() - start)
        print(f"[{elapsed}s] esperando {len(pending)}/{len(logs)} descarga(s)...")
        for log in sorted(pending):
            try:
                last = open(log).readlines()[-1].strip()
            except (FileNotFoundError, IndexError):
                last = "(sin datos aun)"
            print(f"   {log}: {last}")
        time.sleep(poll)
    print("Todas las descargas terminaron.")

wait_for_downloads(DL_LOGS)


## 12. Extraer los pesos de MeToken (ya descargados)

In [ ]:
import zipfile
zipfile.ZipFile('/tmp/pretrained_model.zip').extractall(REPO + '/MeToken')
!ls {REPO}/MeToken/pretrained_model/
!ls -la {REPO}/DeepPTMPred/esm/checkpoints/ {REPO}/EMNgly/esm/checkpoints/ {REPO}/EMNgly/checkpoints/


## 14. Input

Por defecto usa el caso de prueba ya incluido en el repo (`p53_P04637.pdb`, panel de
validación biológica). Para usar tu propia proteína, subí un FASTA o PDB/mmCIF a
`inputs/` con la celda de abajo (opcional).


In [ ]:
from google.colab import files

subir_propio = False  #@param {type:"boolean"}

if subir_propio:
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f"{REPO}/inputs/{fname}")
    input_file = f"{REPO}/inputs/{list(uploaded.keys())[0]}"
else:
    input_file = f"{REPO}/inputs/p53_P04637.pdb"

print("Input:", input_file)


## 15. Correr el pipeline (Fases 1 -> 1.5 -> 2 -> 3 -> 3b)

In [ ]:
%cd {REPO}
!python pipeline.py --input {input_file} --output-dir outputs


## 16. Resultados

In [ ]:
import glob, pandas as pd

report = sorted(glob.glob(f"{REPO}/outputs/*_ptm_sites.csv"), key=os.path.getmtime)[-1]
df = pd.read_csv(report)
print(f"{len(df)} sitio(s) en {report}")
df.sort_values("posicion").head(30)


In [ ]:
import shutil
shutil.copy(report, f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")
print("Reporte respaldado en Drive:", f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")


In [ ]:
resumen = []
for report in sorted(glob.glob(f"{REPO}/outputs/*_ptm_sites.csv")):
    df = pd.read_csv(report)
    resumen.append({
        "proteina": os.path.basename(report).replace("_ptm_sites.csv", ""),
        "sitios": len(df),
        "consenso": int(df["consenso"].sum()) if "consenso" in df.columns else None,
    })
pd.DataFrame(resumen).sort_values("proteina")


## Siguiente paso: Fase 3c (local)

Este notebook deja el reporte de fase 3 con columnas `fase_a_*` = `no_disponible`
(esperado, `FASE_A_ENABLED=false`). Para completar el modelado estructural real:

1. Descarga `report` (o el CSV de `outputs_backup/` en Drive) a tu máquina local.
2. Corre localmente `python pipeline.py --input <mismo archivo>` con el entorno
   `deepptmpred` (PyRosetta) ya instalado -- ahora arranca directo en fase 3c porque las
   fases 1-3b ya están resuelta acá.
